# 02 — Belief propagation on the chain

Companion to **sections 7–10 of `main.pdf`**.

The posterior factorises as prior $\times$ transitions $\times$ local evidence;
its factor graph is a **caterpillar tree**, so sum–product BP is *exact* and
costs $O(K)$ instead of $O(K^3)$.

We use **Convention A** (`main.pdf` Prop. 11): the forward message into $a_k$
carries the evidence $x_0..x_{k-1}$ *only*; the backward message carries
$x_{k+1}..x_{K-1}$ *only*; the local evidence enters once, at combination time.
This is the bookkeeping under which
$$p(a_k|x)\propto \mu_{\to k}(a_k)\;\psi^{(k)}_{\rm ob}(a_k)\;\mu_{\leftarrow k}(a_k)$$
is correct *as written* — the audit caught an earlier version that absorbed the
evidence into the forward message and silently double-counted every $x_k$.

In [1]:
import sys
sys.path.insert(0, 'code')
import numpy as np
np.set_printoptions(precision=6, suppress=True)
from ar1_utils import (ar1_covariance, ou_params, gaussian_posterior,
                       joint_score_matrix)
from bp_score import (bp_posterior, bp_score, _forward_pass, _backward_pass,
                      _evidence_in_a, _gaussian_product)
print('BP code loaded')

BP code loaded


## 1. The $K=3$ worked example — every message, numerically

Same instance as `main.pdf` section 9.3: $\alpha=0.7$, $t=0.5$,
$x=(1.2,\,-0.4,\,0.8)$.  The observation factor, read as a function of $a_k$,
is the Gaussian *pseudo-observation*
$\mathcal N(a_k;\, e^{t}x_k,\, \Delta_t e^{2t})$ — precision $e^{-2t}/\Delta_t$
(the 'pinning strength' of the tilted-measure picture).

In [2]:
K3, alpha3, t3 = 3, 0.7, 0.5
x3 = np.array([1.2, -0.4, 0.8])
sig_eta = np.sqrt(1 - alpha3**2)
mu3, Delta3 = ou_params(t3)
print(f'e^-t = {mu3:.6f}   Delta_t = {Delta3:.6f}   sigma_eta^2 = {1-alpha3**2:.2f}')
print(f'evidence precision e^-2t/Delta = {mu3*mu3/Delta3:.6f}')

m_to, v_to = _forward_pass(x3, t3, alpha3, sig_eta, 0.0, 1.0)
m_lr, v_lr = _backward_pass(x3, t3, alpha3, sig_eta)

print('\nFORWARD  messages mu_->k  (carry x_0..x_{k-1}, Convention A):')
for k in range(K3):
    print(f'   k={k}:  mean={m_to[k]:+.6f}   var={v_to[k]:.6f}')
print('BACKWARD messages mu_<-k  (carry x_{k+1}..x_{K-1}):')
for k in range(K3):
    vv = 'flat (lambda=0)' if np.isinf(v_lr[k]) else f'var={v_lr[k]:.6f}'
    print(f'   k={k}:  mean={m_lr[k]:+.6f}   {vv}')

e^-t = 0.606531   Delta_t = 0.632121   sigma_eta^2 = 0.51
evidence precision e^-2t/Delta = 0.581977

FORWARD  messages mu_->k  (carry x_0..x_{k-1}, Convention A):
   k=0:  mean=+0.000000   var=1.000000
   k=1:  mean=+0.509486   var=0.819739
   k=2:  mean=+0.092348   var=0.781939
BACKWARD messages mu_<-k  (carry x_{k+1}..x_{K-1}):
   k=0:  mean=+0.054410   var=3.585865
   k=1:  mean=+1.884253   var=4.547514
   k=2:  mean=+0.000000   flat (lambda=0)


In [3]:
print('COMBINATION  forward x evidence x backward at each node:')
for k in range(K3):
    m_ev, v_ev = _evidence_in_a(x3[k], t3)
    m_a, v_a = _gaussian_product(m_to[k], v_to[k], m_ev, v_ev)
    m_b, v_b = _gaussian_product(m_a, v_a, m_lr[k], v_lr[k])
    print(f'   posterior a_{k}:  mean={m_b:+.6f}   var={v_b:.6f}')

mu_post, Sig_post = gaussian_posterior(x3, t3, ar1_covariance(K3, alpha3), alpha3)
print('\nmatrix posterior mean      =', mu_post)
print('matrix posterior variances =', np.diag(Sig_post))
print('\n-> identical: BP recombination = exact matrix posterior.')
S3 = (mu3*mu_post - x3)/Delta3
print('score via Tweedie =', S3)
print('note S_1 > 0 although x_1 = -0.4 < 0: the neighbours pull the posterior')
print('mean of a_1 positive -- a purely JOINT effect (main.pdf, end of sec. 9).')

COMBINATION  forward x evidence x backward at each node:
   posterior a_0:  mean=+0.626915   var=0.537389
   posterior a_1:  mean=+0.322520   var=0.494614
   posterior a_2:  mean=+0.475974   var=0.537389

matrix posterior mean      = [0.626915 0.32252  0.475974]
matrix posterior variances = [0.537389 0.494614 0.537389]

-> identical: BP recombination = exact matrix posterior.
score via Tweedie = [-1.296836  0.942254 -0.808876]
note S_1 > 0 although x_1 = -0.4 < 0: the neighbours pull the posterior
mean of a_1 positive -- a purely JOINT effect (main.pdf, end of sec. 9).


## 2. Hand-check of one forward step (transparency)

Forward step $0\to1$, exactly as in the paper:
1. **update**: combine prior $(0,1)$ with pseudo-observation at node 0 —
   precisions add;
2. **predict**: push through the transition — mean $\times\alpha$, variance
   $\alpha^2 v + \sigma_\eta^2$.

This is one predict–update cycle of the **Kalman filter**; the backward sweep
plus combination is the **RTS smoother**.

In [4]:
lam_c = 1.0 + mu3*mu3/Delta3                      # combined precision at node 0
v_c   = 1.0/lam_c
m_c   = v_c*(0.0 + (mu3/Delta3)*x3[0])
print(f'update : combined (m, v) at node 0 = ({m_c:.6f}, {v_c:.6f})')
m_pred = alpha3*m_c
v_pred = alpha3**2*v_c + (1-alpha3**2)
print(f'predict: message into a_1        = ({m_pred:.6f}, {v_pred:.6f})')
print(f'code   : mu_->1                  = ({m_to[1]:.6f}, {v_to[1]:.6f})   match')

update : combined (m, v) at node 0 = (0.727837, 0.632121)
predict: message into a_1        = (0.509486, 0.819739)
code   : mu_->1                  = (0.509486, 0.819739)   match


## 3. BP $=$ exact score, across the parameter space

The same comparison the audit runs as its section 5: 80 configurations
($K\in\{2,3,5,8,16\}$, $\alpha\in\{0.2,0.5,0.9,-0.4\}$,
$t\in\{0.05,0.3,1,3\}$), random $x$.  Gaussian messages are **exact closure**
(product of Gaussians is Gaussian, Gaussian through a linear transition is
Gaussian) — no central limit theorem anywhere, so exactness holds for any node
degree, on this degree-2 chain included (`main.pdf` Th. 13).

In [5]:
err = 0.0
for Kk in (2, 3, 5, 8, 16):
    for aa in (0.2, 0.5, 0.9, -0.4):
        for tt in (0.05, 0.3, 1.0, 3.0):
            xx = np.random.default_rng(Kk).standard_normal(Kk)
            err = max(err, np.max(np.abs(
                bp_score(xx, tt, aa)
                - joint_score_matrix(xx, tt, ar1_covariance(Kk, aa), aa))))
print('max |S_BP - S_exact| over 80 configurations =', err)
print('-> BP IS the score, computed through the tridiagonal posterior in O(K).')

max |S_BP - S_exact| over 80 configurations = 1.4210854715202004e-14
-> BP IS the score, computed through the tridiagonal posterior in O(K).


## Summary

* Convention A avoids the double-counting trap (the audit is what caught it).
* Forward sweep $=$ Kalman filter; combination $=$ RTS smoother.
* All messages exactly Gaussian by algebraic closure — *not* a CLT.
* BP score $=$ matrix score to $\sim 10^{-14}$ over the full parameter sweep.

Continue with **`03_amp_and_experiments.ipynb`** for the closed-form bulk
analysis, the AMP comparison and the two experiments.